# DLBDSNNDL01 – Neural Nets and Deep Learning

# Task 1: Accelerating Deep Network Training via Batch Normalization

This notebook presents the solution to Task 1 of the course DLBDSNNDL01.  
The task consists in addressing three different issues:

1. Explain how a deep neural network is trained and which challenge is addressed with batch normalization.
2. Describe how batch normalization is implemented, also for convolutional neural networks (CNNs).
3. Explain how the acceleration of the training can be measured and how batch normalization performs with respect to, at least one, competing approach from the literature.

The presentation that summarizes these results is not part of the repository.

## 1. Introduction

A machine learning model transforms its input data into meaningful outputs, a process that is "learned" from exposure to known examples of inputs and outputs. The central problem in machine learning and deep learning is to learn useful representations of the input data that get us closer to the expected output. _Learning_ is the the automatic search for data transformations that produce useful representations of the input data, guided by some feedback signal.

Deep learning represents a new approach on learning repsentations from data that emphasizes learning succesive layers of increasingly meaningful representations. These layered representations of data are learned via models called neural networks.

## 2. How does a neural network _learn_?

The behavior of a layer is determined by its trainable parameters, such as its _weights_ and _biases_. _Learning_ means to find the set of parameter values for all layers in a network such that the network correctly maps example inputs to their associated targets.

To measure how well a neural network learns, we need to measure how far its output is from the expected value. This is done using the _loss function_. The loss function takes the predictions of the network and the true targets and calculates a score that describes how well the network is performing. The goal of learning is to reduce the value of the loss by using its value to adjust the value of the weights of the network.

This is achieved using _gradient descent_, which iteratively adjusts the value of the network's parameters during training. The idea of gradient descent is to adjust the parameters in the direction that minimizes the loss function.

$$
\boxed{
\theta \leftarrow \theta - \eta \nabla_{\theta}\mathcal{L}(\theta)
}
$$

where

$$
\begin{aligned}
\theta & \rightarrow \text{the vector containing all trainable parameters} \\
\nabla_{\theta}\mathcal{L}(\theta) & \rightarrow \text{the gradient of the loss with respect to the parameters} \\
\mathcal{L}(\theta) & \rightarrow \text{the loss function} \\
\eta & \rightarrow \text{the learning rate}
\end{aligned}
$$

Mathematically, the gradient is the vector of partial derivatives of the loss function with respect to the network parameters. It tells us the direction in which the loss increases most rapidly. Therefore, we move in the oposite direction in order to minimize the loss.

The algorithm for efficiently calculating the gradients of the loss function with respect to the network parameters is called _backpropagation_.

In simple terms, backpropagation is the repeated application of the _chain rule_ of calculus to obtain the partial derivatives of the loss function with respect to each of the parameters of the network.  

For example, assuming we have a small network with three layers, we can calculate the partial derivative of the loss function with respect to the weights of the third layer as:

$$
\frac{\partial \mathcal{L}}{\partial W^{[3]}} = 
\frac{\partial \mathcal{L}}{\partial A^{[3]}} \cdot
\frac{\partial A^{[3]}}{\partial Z^{[3]}} \cdot
\frac{\partial Z^{[3]}}{\partial W^{[3]}}
$$

where

$$
\begin{aligned}
\mathcal{L} & \rightarrow \text{the loss function} \\
A^{[3]} & \rightarrow \text{the activation/output of the third layer} \\
Z^{[3]} & \rightarrow \text{the pre-activation of the third layer} \\
\end{aligned}
$$

The learning process then encompasses the following steps:

1. Forward pass: propagate the input through the network to calculate the output using the current parameters.
2. Loss calculation: compare the network's prediction with the target using the loss function.
3. Backward pass: propagate backwards through the network to calculate the gradients of the loss with respect to the parameters.
4. Parameter update: adjust the network parameters using the calculated gradients and an optimization algorithm such as gradient descent.

## 3. Batch normalization

Training a deep neural network is difficult due to the fact that the distribution of each layer's input changes during training, as the parameters of previous layers change. Ioffe and Szegedy (2015) attributed this difficulty to what they called _internal covariate shift_. This slows down training, by requiring lower learning rates, careful parameter initialization and makes it hard to train networks with saturating nonlinearities such as the _sigmoid function_.

Ioffe and Szegedy introduced batch normalization as a method to reduce the internal covariate shift, which consists in fixing the distribution of the layer inputs as training progresses and which showed that accelerated training of deep neural networks was possible. The authors also showed that BN made it possible to use higher learning rates and that fewer training steps were necessary to reach a certain accuracy. They also observed that BN had a regularizing effect that could sometimes reduce the need for Dropout.

Later work by Balestriero and Baraniuk (2022) provided new insights into how BN boosts network optimization and inference performance. They provide a geometric explanation of the advantages of BN. ReLU networks partition their input space into regions where the network behaves as an affine function. BN therefore produces a data-adapted partition even before gradient-based learning has taken place, which they interpret as a _smart initilization_. They additionally showed that the randomness of the mini-batch statistics introduces some dropout-like perturbation that can improve generalization.

### 3.1 How is batch normalization implemented?

Batch normalization consists in adding an operation just before or after the _activation_ function of each hidden layer, Ioffe and Szegedy (2015) argued in favor of adding the BN layer before the activation, so that is the convention used in the following experiments. This operation simply zero-centers and normalizes each layer input (activation) and then scales and shifts the result using two new learnable parameters per normalized feature:

$$
\begin{aligned}
\gamma & \rightarrow \text{for scaling} \\
\beta & \rightarrow \text{for shifting}
\end{aligned}
$$

In order to zero-center and normalize the inputs, BN estimates the input's _mean_ and _standard deviation_. It does this by evaluating the mean and std of the input over the current _mini-batch_. The batch normalization algorithm can be summarized step by step as follows

$$
\begin{aligned}
\text{1. } &\mathbf{\mu}_{B} = \frac{1}{m_{B}}\sum_{i=1}^{m_B} \mathbf{x}^{(i)} && \rightarrow \text{vector of input means}\\
\\
\text{2. } &\mathbf{\sigma}_{B}^2 = \frac{1}{m_{B}}\sum_{i=1}^{m_B} (\mathbf{x}^{(i)} - \mathbf{\mu}_{B})^2 && \rightarrow \text{vector of input standard deviations}\\
\\
\text{3. } &\mathbf{\hat x}^{(i)} = \frac{\mathbf{x}^{(i)} - \mathbf{\mu}_{B}}{\sqrt{(\mathbf{\sigma}_{B}^2 + \epsilon)}} && \rightarrow \text{vector of zero-centered and normalized inputs for instance } \mathcal{i}\\
\\
\text{4. } &\mathbf{z}^{(i)} = \mathbf{\gamma} \mathbf{\hat x}^{(i)} + \mathbf{\beta} && \rightarrow \text{the output of BN. A rescaled and shifted version of the inputs}
\end{aligned}
$$

where

$$
\begin{aligned}
m_B & \rightarrow \text{ is the number of instances in the mini-batch}\\
\epsilon & \rightarrow \text{ the smoothing term. It is a tiny number that avoid zero division}\\
\otimes & \rightarrow \text{ element-wise multiplication}
\end{aligned}
$$

#### 3.1.1 Convolutional Neural Networks

For CNNs, batch normalization works in the same way as it does for FNNs with the difference that the mean and variance used for normalization are calculated for all instances in the mini-batch and all spatial locations $(H \times W)$ for each feature map or channel $C$.

Assuming we have a mini-batch of size $(\text{B} \times \text{W} \times \text{H} \times \text{C})$, we can summarize the implementation of the BN algorithm for CNNs as follows:

$$
\begin{aligned}
\text{1. } &\mathbf{\mu}_{C} = \frac{1}{BHW}\sum_{b=1}^{B}\sum_{h=1}^{H}\sum_{w=1}^{W} \mathbf{X}_{b,h,w,c} && \rightarrow \text{vector of input means per channel}\\
\\
\text{2. } &\mathbf{\sigma}_{C}^2 = \frac{1}{BHW}\sum_{b=1}^{B}\sum_{h=1}^{H}\sum_{w=1}^{W}(\mathbf{X}_{b,h,w,c} - \mathbf{\mu}_{C})^2 && \rightarrow \text{vector of input variances per channel}\\
\\
\text{3. } &\mathbf{\hat X}_{b,h,w,c} = \frac{\mathbf{X}_{b,h,w,c} - \mathbf{\mu}_{C}}{\sqrt{(\mathbf{\sigma}_{C}^2 + \epsilon)}} && \rightarrow \text{tensor of normalized features per channel C}\\
\\
\text{4. } &\mathbf{Z}_{b,h,w,c} = \mathbf{\gamma_C} \mathbf{\hat X}_{b,h,w,c} + \mathbf{\beta_C} && \rightarrow \text{the output of BN. A rescaled and shifted version of the inputs}
\end{aligned}
$$

The difference:
- for FNNs, we compute the mean and variance per feature across the batch
- for CNNs, we compute the mean and variance per channel using all spatial locations $(H \times W)$ and all example in the mini-batch $B$

For example, assuming an FNN and a 2D dataset with 28 features, we have $(\text{mini-batch} \times \text{features})=(B \times 28)$ this results in 28 means + 28 variances, each feature gets its own statistics. For a CNN and a 4D dataset of shape $(B \times H \times W \times C) = (64 \times 28 \times 28 \times 3)$ this results in 3 means and 3 variances, one for each channel.

## 4. Acceleration of the training process

To illustrate the effects of batch normalization several experiments according ot the table below have been performed.

<table>
    <tr>
        <th>Experiment</th>
        <th>Architecture</th>
        <th>Normalization</th>
        <th>Optimizer</th>
        <th>Learning rate</th>
        <th>Remarks</th>
    </tr>
    <tr>
        <td>FNN-1</td>
        <td>FNN</td>
        <td>None</td>
        <td>SGD with momentum = 0.9</td>
        <td>0.001</td>
        <td>Baseline</td>
    <tr>
        <td>FNN-2</td>
        <td>FNN</td>
        <td>Batch Normalization</td>
        <td>SGD with momentum = 0.9</td>
        <td>0.001</td>
        <td></td>
    </tr>
    <tr>
        <td>FNN-3</td>
        <td>FNN</td>
        <td>Layer Normalization</td>
        <td>SGD with momentum = 0.9</td>
        <td>0.001</td>
        <td></td>
    </tr>
</table>

The results show that batch normalization indeed accelerates the training of a neural network in the sense that it help achieves a faster convergence, it does not make training faster by reducing the time. As it can be observed in the training logs, training for the models with batch normalization taker longer for each epoch due to the additional operation necessary to learn the batch-norm parameters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist, fashion_mnist, california_housing
from tensorflow.keras import models, layers, Input
from tensorflow.keras.utils import to_categorical

# These are utilities for data visualization and plotting
import utils
from utils import ActivationLogger

In [ ]:
# get the dataset
fashion_mnist = fashion_mnist.load_data()

In [ ]:
# create the train and test set
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist

In [ ]:
# Shape of the full training set
X_train_full.shape

In [ ]:
# Shape of the test set
X_test.shape

In [ ]:
# Show an instance of the dataset
idx = 162
training_example = X_train_full[idx]
label = y_train_full[idx]
fig, ax = plt.subplots()
utils.visualize_fmnist(training_example, label, ax)

In [ ]:
train_size = 50_000 # Thus 10_000 samples for validation

# Create the validation set and normalize the input data
X_train, y_train = X_train_full[:train_size]/255, y_train_full[:train_size]
X_valid, y_valid = X_train_full[train_size:]/255, y_train_full[train_size:]
X_test = X_test / 255

print(f"Shape training data {X_train.shape}")
print(f"Data Type training data {X_train.dtype}")

print(f"Shape training labels {y_train.shape}")
print(f"Data Type training labels {y_train.dtype}")

print(f"Shape validation data {X_valid.shape}")
print(f"Data Type training data {X_valid.dtype}")

print(f"Shape validation labels {y_valid.shape}")
print(f"Data Type validation labels {y_valid.dtype}")

### FNN without batch normalization - Baseline

In [ ]:
# This model was created using the Functional API to expose the Input Layer so it can be used
# with the ActivationLogger Callback - see utils.py

# Use this to clear keras history
tf.keras.backend.clear_session() 

# Using the same random seed makes comparison easier
tf.random.set_seed(42)

L0_inputs  = layers.Input(shape=[28,28], name="L0_inputs")
L0_flatten = layers.Flatten(name="L0_flatten")(L0_inputs)

n_layers = 6

A = L0_flatten
for n in range(1, n_layers+1):
    if n == n_layers: # the last layer
        n_neurons          = 10
        activation         = "softmax"
        name_Z             = f"L{n}_output_Z"
        name_BN            = f"L{n}_output_BN"
        name_A             = f"L{n}_output_A"
        kernel_initializer = "glorot_uniform"
        use_bias           = True
    
    if n != n_layers: # if it is not the las layer
        n_neurons          = 100 # better to increase n_layers than to increase n_units
        activation         = "sigmoid"
        name_Z             = f"L{n}_Z"
        name_BN            = f"L{n}_BN"
        name_A             = f"L{n}_A"
        kernel_initializer = "glorot_uniform"
        use_bias           = True

    Z = layers.Dense(units=n_neurons, kernel_initializer=kernel_initializer, use_bias=use_bias, name=name_Z)(A)
    A  = layers.Activation(activation=activation, name=name_A)(Z)

# This is the model that is used for the predictions
no_bn_model = keras.Model(
    inputs=L0_inputs,
    outputs=A
)

In [ ]:
no_bn_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True),
    metrics=["accuracy"]
)

In [ ]:
no_bn_model.summary()

In [ ]:
# The activation_model exposes the results from the intermediate activations of the model 
# The idea is for this model to be used in in a callback during training
activation_model_no_bn = keras.Model(
    inputs=no_bn_model.input,
    outputs=[no_bn_model.get_layer(f"L{n}_{t}").output for n in range(1,n_layers) for t in ("Z", "A")]
)

In [ ]:
activation_model_no_bn.summary()

In [ ]:
epochs = 500
batch_size = 64
X_probe = X_train[:batch_size*10]

In [ ]:
# If batch_site == 64 then X_probe will have a size of 640 samples for which the intermediate activations will be collected
# parameter initilization

initial_activations = []

for i in range(0, len(X_probe), batch_size):
    X_batch = X_probe[i:i+batch_size]
    # Before training, the activation model has only the weights with which the no_bn_model was initialized
    # Make predictions using the initialized weights
    outputs = activation_model_no_bn(X_batch, training=True)

    initial_activations.append(
        [x.numpy() for x in outputs]
    )

# Number of analized mini-batches
len(initial_activations)

In [ ]:
# How many activations does every mini-batch have
len(initial_activations[0])

In [ ]:
# Train the model
activation_logger = ActivationLogger(
    X_probe,
    activation_model=activation_model_no_bn,
    batch_size=batch_size,
    every_n_epochs=10
)
activation_logger.set_init_activations(initial_activations) # include the initial activation in the logger to plot later

In [ ]:
history = no_bn_model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_valid, y_valid),
    callbacks=[
        activation_logger,
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=True)
    ]
)

In [ ]:
df_no_bn = pd.DataFrame(history.history)

fig, axs = plt.subplots(figsize=(10,6), nrows=2, ncols=1, sharex=True)
for ax in axs:
    ax.set_xticks(np.arange(0,280,10), np.arange(0,280,10))
    ax.set_xlim([0,270])
utils.plot_train_and_val_error(df_no_bn, axs)
fig.tight_layout()

In [ ]:
train_loss, train_accuracy = no_bn_model.evaluate(X_train, y_train)
print(f"Train accuracy {train_accuracy:.2f}")
print(f"Train loss {train_loss:.2f}")

In [ ]:
test_loss, test_accuracy = no_bn_model.evaluate(X_test, y_test)
print(f"Test accuracy {test_accuracy:.2f}")
print(f"Test loss {test_loss:.2f}")

In [ ]:
activation_names = [no_bn_model.get_layer(f"L{n}_{t}").name for n in range(1,n_layers) for t in ("Z", "A")]
#epoch_steps = [60,190,250] # uncomment to generate picture for presentation
#figsize = (20,6) # uncomment to generate picture for presentation
epoch_steps = list(activation_logger.activations.keys())
figsize = (36,48)
fig, axs = plt.subplots(nrows=len(epoch_steps), ncols=len(activation_names), figsize=figsize, sharey=False)
for i in range(len(epoch_steps)):
    for mini_batch in range(0,10):
        activations = activation_logger.activations[epoch_steps[i]][mini_batch] # epoch and minibatch
        for activation_name, activation in zip(activation_names, activations):
            axs[i, activation_names.index(activation_name)].set_title(f"Epoch {epoch_steps[i]}, {activation_name}", fontsize=10)
            axs[i, activation_names.index(activation_name)].hist(activation.flatten(), bins=30, alpha=0.2, density=True, label=f"epoch{epoch_steps[i]}")
fig.tight_layout()

In [ ]:
all_activations = []
for i in range(len(epoch_steps)):
    for mb in range(0,10):
        activations = activation_logger.activations[epoch_steps[i]][mb] # epoch and minibatch
        row = [epoch_steps[i], mb]
        for activation in activations:
            row.extend([activation.mean(), activation.std(), activation.min(), activation.max()])
        all_activations.append(row)

In [ ]:
columns = ["epoch","mb"]
columns_base_name = [layer.name for layer in activation_model_no_bn.layers][2:]
for col in columns_base_name:
    columns.extend([f"{col}_mean", f"{col}_std", f"{col}_min", f"{col}_max"])

In [ ]:
df = pd.DataFrame(all_activations, columns=columns)
df

In [ ]:
df.groupby("epoch").mean().round(4).T

### FNN with batch normalization

In [ ]:
tf.keras.backend.clear_session() 

tf.random.set_seed(42)

L0_inputs  = layers.Input(shape=[28,28], name="L0_inputs")
L0_flatten = layers.Flatten(name="L0_flatten")(L0_inputs)

#n_layers = 6 # use the same number of layer as in the baseline model

A = L0_flatten
for n in range(1, n_layers+1):
    if n == n_layers: # the last layer
        n_neurons          = 10
        activation         = "softmax"
        name_Z             = f"L{n}_output_Z"
        name_BN            = f"L{n}_output_BN"
        name_A             = f"L{n}_output_A"
        kernel_initializer = "glorot_uniform"
        use_bias           = False
    
    if n != n_layers: # if it is not the las layer
        n_neurons          = 100 # better to increase n_layers than to increase n_units
        activation         = "sigmoid"
        name_Z             = f"L{n}_Z"
        name_BN            = f"L{n}_BN"
        name_A             = f"L{n}_A"
        kernel_initializer = "glorot_uniform"
        use_bias           = False

    Z  = layers.Dense(units=n_neurons, kernel_initializer=kernel_initializer, use_bias=use_bias, name=name_Z)(A)
    BN = layers.BatchNormalization(name=name_BN)(Z)
    A  = layers.Activation(activation=activation, name=name_A)(BN)

# This is the model that is used for the predictions
bn_model = keras.Model(
    inputs=L0_inputs,
    outputs=A
)

In [ ]:
bn_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True),
    metrics=["accuracy"]
)

In [ ]:
bn_model.summary()

In [ ]:
activation_model_bn = keras.Model(
    inputs=bn_model.input,
    outputs=[bn_model.get_layer(f"L{n}_{t}").output for n in range(1,n_layers) for t in ("Z", "BN", "A")]
)

In [ ]:
activation_model_bn.summary()

In [ ]:
initial_activations_bn = []

for i in range(0, len(X_probe), batch_size):
    X_batch = X_probe[i:i+batch_size]
    # Before training, the activation model has only the weights with which the no_bn_model was initialized
    # Make predictions using the initialized weights
    outputs = activation_model_bn(X_batch, training=True)

    initial_activations_bn.append(
        [x.numpy() for x in outputs]
    )

# Number of analized mini-batches
len(initial_activations_bn)

In [ ]:
activation_logger_bn = ActivationLogger(
    X_probe,
    activation_model=activation_model_bn,
    batch_size=batch_size,
    every_n_epochs=10
)
activation_logger_bn.set_init_activations(initial_activations_bn)

In [ ]:
history_bn = bn_model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_valid, y_valid),
    callbacks=[
        activation_logger_bn,
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=True)
    ]
)

In [ ]:
df_bn = pd.DataFrame(history_bn.history)

fig, axs = plt.subplots(figsize=(10,6), nrows=2, ncols=1, sharex=True, sharey=False)
for ax in axs:
    ax.set_xticks(np.arange(0,25,1), np.arange(0,25,1))
    ax.set_xlim([0,25])
utils.plot_train_and_val_error(df_bn, axs)
fig.tight_layout()

In [ ]:
train_loss, train_accuracy = bn_model.evaluate(X_train, y_train)
print(f"Train accuracy {train_accuracy:.2f}")
print(f"Train loss {train_loss:.2f}")

In [ ]:
val_loss, val_accuracy = bn_model.evaluate(X_test, y_test)
print(f"Validation accuracy {val_accuracy:.2f}")
print(f"Validation loss {val_loss:.2f}")

In [ ]:
activation_names = [bn_model.get_layer(f"L{n}_{t}").name for n in range(1,n_layers) for t in ("Z", "BN", "A")]
epoch_steps = list(activation_logger_bn.activations.keys())
figsize=(36,8)

fig, axs = plt.subplots(nrows=len(epoch_steps), ncols=len(activation_names), figsize=figsize, sharey=False)
for i in range(len(epoch_steps)):
    for mini_batch in range(0,10):
        activations = activation_logger_bn.activations[epoch_steps[i]][mini_batch] # epoch and minibatch
        for activation_name, activation in zip(activation_names, activations):
            axs[i, activation_names.index(activation_name)].set_title(f"Epoch {epoch_steps[i]}, {activation_name}")
            axs[i, activation_names.index(activation_name)].hist(activation.flatten(), bins=30, alpha=0.2, density=True, label=f"epoch{epoch_steps[i]}")
fig.tight_layout()

In [ ]:
all_activations = []
for i in range(len(epoch_steps)):
    for mb in range(0,10):
        activations = activation_logger_bn.activations[epoch_steps[i]][mb] # epoch and minibatch
        row = [epoch_steps[i], mb]
        for activation in activations:
            row.extend([activation.mean(), activation.std(), activation.min(), activation.max()])
        all_activations.append(row)

In [ ]:
columns_bn = ["epoch","mb"]
columns_base_name_bn = [layer.name for layer in activation_model_bn.layers][2:]
for col in columns_base_name_bn:
    columns_bn.extend([f"{col}_mean", f"{col}_std", f"{col}_min", f"{col}_max"])

In [ ]:
df_bn = pd.DataFrame(
    all_activations,
    columns=columns_bn
)
df_bn

In [ ]:
df_bn.groupby("epoch").mean().round(4).T

### FNN with Layer Normalization

In [ ]:
tf.keras.backend.clear_session() 

tf.random.set_seed(42)

L0_inputs  = layers.Input(shape=[28,28], name="L0_inputs")
L0_flatten = layers.Flatten(name="L0_flatten")(L0_inputs)

#n_layers = 6 # use the same number of layer as in the baseline model

A = L0_flatten
for n in range(1, n_layers+1):
    if n == n_layers: # the last layer
        n_neurons          = 10
        activation         = "softmax"
        name_Z             = f"L{n}_output_Z"
        name_LN            = f"L{n}_output_LN"
        name_A             = f"L{n}_output_A"
        kernel_initializer = "glorot_uniform"
        use_bias           = False
    
    if n != n_layers: # if it is not the las layer
        n_neurons          = 100 # better to increase n_layers than to increase n_units
        activation         = "sigmoid"
        name_Z             = f"L{n}_Z"
        name_LN            = f"L{n}_LN"
        name_A             = f"L{n}_A"
        kernel_initializer = "glorot_uniform"
        use_bias           = False

    Z  = layers.Dense(units=n_neurons, kernel_initializer=kernel_initializer, use_bias=use_bias, name=name_Z)(A)
    LN = layers.LayerNormalization(name=name_LN)(Z)
    A  = layers.Activation(activation=activation, name=name_A)(LN)

# This is the model that is used for the predictions
ln_model = keras.Model(
    inputs=L0_inputs,
    outputs=A
)

In [ ]:
ln_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True),
    metrics=["accuracy"]
)

In [ ]:
ln_model.summary()

In [ ]:
activation_model_ln = keras.Model(
    inputs=ln_model.input,
    outputs=[ln_model.get_layer(f"L{n}_{t}").output for n in range(1,n_layers) for t in ("Z", "LN", "A")]
)

In [ ]:
activation_model_ln.summary()

In [ ]:
initial_activations_ln = []

for i in range(0, len(X_probe), batch_size):
    X_batch = X_probe[i:i+batch_size]
    outputs = activation_model_ln(X_batch, training=True)
    initial_activations_ln.append(
        [x.numpy() for x in outputs]
    )

# Number of analized mini-batches
len(initial_activations_ln)

In [ ]:
# Train the model
activation_logger_ln = ActivationLogger(
    X_probe,
    activation_model=activation_model_ln,
    batch_size=batch_size,
    every_n_epochs=10
)
activation_logger_ln.set_init_activations(initial_activations_ln)

In [ ]:
history_ln = ln_model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_valid, y_valid),
    callbacks=[
        activation_logger_ln,
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=True)
    ]
)

In [ ]:
df_ln = pd.DataFrame(history_ln.history)

fig, axs = plt.subplots(figsize=(10,6), nrows=2, ncols=1, sharex=True)
for ax in axs:
    ax.set_xticks(np.arange(0,50,1), np.arange(0,50,1))
    ax.set_xlim([0,50])
utils.plot_train_and_val_error(df_ln, axs)
fig.tight_layout()

In [ ]:
train_loss, train_accuracy = ln_model.evaluate(X_train, y_train)
print(f"Train accuracy {train_accuracy:.2f}")
print(f"Train loss {train_loss:.2f}")

In [ ]:
val_loss, val_accuracy = ln_model.evaluate(X_test, y_test)
print(f"Validation accuracy {val_accuracy:.2f}")
print(f"Validation loss {val_loss:.2f}")

In [ ]:
activation_names = [ln_model.get_layer(f"L{n}_{t}").name for n in range(1,n_layers) for t in ("Z", "LN", "A")]
epoch_steps = list(activation_logger_ln.activations.keys())

fig, axs = plt.subplots(nrows=len(epoch_steps), ncols=len(activation_names), figsize=(36,12), sharey=False)
for i in range(len(epoch_steps)):
    for mini_batch in range(0,10):
        activations = activation_logger_ln.activations[epoch_steps[i]][mini_batch] # epoch and minibatch
        for activation_name, activation in zip(activation_names, activations):
            axs[i, activation_names.index(activation_name)].set_title(f"Epoch {epoch_steps[i]}, {activation_name}")
            axs[i, activation_names.index(activation_name)].hist(activation.flatten(), bins=30, alpha=0.2, density=True, label=f"epoch{epoch_steps[i]}")
fig.tight_layout()

In [ ]:
all_activations = []
for i in range(len(epoch_steps)):
    for mb in range(0,10):
        activations = activation_logger_ln.activations[epoch_steps[i]][mb] # epoch and minibatch
        row = [epoch_steps[i], mb]
        for activation in activations:
            row.extend([activation.mean(), activation.std(), activation.min(), activation.max()])
        all_activations.append(row)

In [ ]:
columns_ln = ["epoch","mb"]
columns_base_name_ln = [layer.name for layer in activation_model_ln.layers][2:]
for col in columns_base_name_ln:
    columns_ln.extend([f"{col}_mean", f"{col}_std", f"{col}_min", f"{col}_max"])

In [ ]:
df_ln = pd.DataFrame(
    all_activations,
    columns=columns_ln
)
df_ln

In [ ]:
df_ln.groupby("epoch").mean().round(4).T

### Baseline vs. Batch Normalization vs. Layer Normalization

In [ ]:
df_error = pd.DataFrame(history.history)

In [ ]:
df_error.columns

In [ ]:
df_error_bn = pd.DataFrame(history_bn.history)

In [ ]:
df_error_bn.columns

In [ ]:
df_error_ln = pd.DataFrame(history_ln.history)

In [ ]:
df_error_ln.columns

In [ ]:
fig, axs = plt.subplots(figsize=(30,20), nrows=2, ncols=1, sharex=True)
title_fontsize = 36
labels_fontsize = 28
axes_fontsize = 28
linewidth = 5
linestyle = "--"

# Accuracy
axs[0].plot(df_error.index,    df_error["accuracy"],        label="Train accuracy no Norm", linewidth=linewidth)
axs[0].plot(df_error.index,    df_error["val_accuracy"],    label="Val accuracy no Norm",   linewidth=linewidth, linestyle=linestyle)
axs[0].plot(df_error_bn.index, df_error_bn["accuracy"],     label="Train accuracy BN",    linewidth=linewidth)
axs[0].plot(df_error_bn.index, df_error_bn["val_accuracy"], label="Val accuracy BN",      linewidth=linewidth, linestyle=linestyle)
axs[0].plot(df_error_ln.index, df_error_ln["accuracy"],     label="Train accuracy LN", linewidth=linewidth)
axs[0].plot(df_error_ln.index, df_error_ln["val_accuracy"], label="Val accuracy LN",      linewidth=linewidth, linestyle=linestyle)
axs[0].set_yticks(np.arange(0,1.1,0.1), [f"{n:.2f}" for n in np.arange(0,1.1,0.1)], fontsize=axes_fontsize)
axs[0].set_title("Training and validation accuracy", fontsize=title_fontsize)
axs[0].set_ylabel("Accuracy", fontsize=labels_fontsize)

axs[1].plot(df_error.index,    df_error["loss"],        label="Train loss no Norm ", linewidth=linewidth)
axs[1].plot(df_error.index,    df_error["val_loss"],    label="Val loss No BN",   linewidth=linewidth, linestyle=linestyle)
axs[1].plot(df_error_bn.index, df_error_bn["loss"],     label="Train loss BN",    linewidth=linewidth)
axs[1].plot(df_error_bn.index, df_error_bn["val_loss"], label="Val loss BN",      linewidth=linewidth, linestyle=linestyle)
axs[1].plot(df_error_ln.index, df_error_ln["loss"],     label="Train loss LN",    linewidth=linewidth)
axs[1].plot(df_error_ln.index, df_error_ln["val_loss"], label="Val loss LN",      linewidth=linewidth, linestyle=linestyle)
axs[1].set_yticks(np.arange(0,3.2,0.2), [f"{n:.2f}" for n in np.arange(0,3.2,0.2)], fontsize=axes_fontsize)
axs[1].set_title("Training and validation loss", fontsize=title_fontsize)
axs[1].set_ylabel("Loss", fontsize=labels_fontsize)
axs[1].set_xlabel("Epochs", fontsize=labels_fontsize)

for ax in axs:
    ax.legend(loc="lower right", fontsize=labels_fontsize)
    ax.grid(True, linewidth=0.5, linestyle=":")
    ax.set_xlim([0,310])
    ax.set_xticks(np.arange(0,320,10), np.arange(0,320,10), fontsize=axes_fontsize)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()